In [1]:
%reset -f
%run setup_notebook.py

from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import numpy as np
from like3.pixel_table import PixelTable
from like3.sourcelist import SourceModel

source_name = 'Mrk 421'
roi_center = SkyCoord(166.1138, 38.2088, unit='deg', frame='icrs')
roi_sources = SourceModel.from_fermi_catalog(
    'v40',
    skydir=roi_center,
    cone_size=1.0,
    query='significance >= 25',
)
pt = PixelTable('files/kerr/toby_v4.fits', source_model=roi_sources)

if source_name in set(roi_sources.source_names):
    target_name = source_name
else:
    target_name = str(roi_sources.source_names[0])
    print(f"'{source_name}' not found in ROI; using '{target_name}'")

src = pt.source_model.find_source(target_name)
loc = pt.localize(target_name, sigma=0.1, verbose=True)

if hasattr(loc, 'dir') and loc.dir is not None:
    src.skydir = loc.dir.coord if hasattr(loc.dir, 'coord') else loc.dir

def _to_skycoord(obj):
    return obj.coord if hasattr(obj, 'coord') else obj

def _scan_delta_ts_grid(loc, ra0, dec0, size_deg=0.4, nside=41):
    dra = np.linspace(-size_deg, size_deg, nside)
    ddec = np.linspace(-size_deg, size_deg, nside)
    cosdec = max(abs(np.cos(np.radians(dec0))), 1e-6)
    delta_ts = np.zeros((nside, nside), dtype=float)
    for iy, dy in enumerate(ddec):
        for ix, dx in enumerate(dra):
            trial = SkyCoord(
                ra=ra0 + dx / cosdec,
                dec=dec0 + dy,
                unit='deg',
                frame='icrs',
            )
            delta_ts[iy, ix] = float(loc.TS(trial))
    return dra, ddec, delta_ts

center = _to_skycoord(src.skydir)
ra0, dec0 = center.ra.deg, center.dec.deg

# Auto-zoom until at least a few samples are above Delta TS = -25.
size_deg = 0.4
nside = 41
for _ in range(6):
    dra, ddec, delta_ts_raw = _scan_delta_ts_grid(loc, ra0, dec0, size_deg=size_deg, nside=nside)
    core_mask = delta_ts_raw > -25.0
    if np.count_nonzero(core_mask) >= 16:
        break
    size_deg *= 0.5
    nside = min(161, nside + 20)

# Keep only the requested Delta TS range: [-25, 0].
delta_ts_grid = np.clip(delta_ts_raw, -25.0, 0.0)
sqrt_minus_delta_ts = np.sqrt(-delta_ts_grid)

xx, yy = np.meshgrid(dra, ddec)
core_mask = np.isfinite(delta_ts_raw) & (delta_ts_raw > -25.0)
if np.any(core_mask):
    x_in = xx[core_mask]
    y_in = yy[core_mask]
    xspan = max(float(x_in.max() - x_in.min()), float(dra[1] - dra[0]))
    yspan = max(float(y_in.max() - y_in.min()), float(ddec[1] - ddec[0]))
    xpad = max(0.01, 0.25 * xspan)
    ypad = max(0.01, 0.25 * yspan)
    xlim = (float(x_in.min() - xpad), float(x_in.max() + xpad))
    ylim = (float(y_in.min() - ypad), float(y_in.max() + ypad))
else:
    xlim = (float(dra.min()), float(dra.max()))
    ylim = (float(ddec.min()), float(ddec.max()))

fig, ax = plt.subplots(figsize=(6, 5))
contours = ax.contourf(
    dra,
    ddec,
    sqrt_minus_delta_ts,
    levels=np.linspace(0.0, 5.0, 21),
    vmin=0.0,
    vmax=5.0,
    cmap='jet_r',
    extend='max',
)
ax.scatter([0], [0], c='white', s=60, marker='*', label='Current source position')

if hasattr(loc, 'dir') and loc.dir is not None:
    loc_sc = _to_skycoord(loc.dir)
    ax.scatter(
        [(loc_sc.ra.deg - ra0) * np.cos(np.radians(dec0))],
        [loc_sc.dec.deg - dec0],
        c='black',
        s=45,
        marker='x',
        label='Localization center',
    )

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Delta RA * cos(dec) [deg]')
ax.set_ylabel('Delta Dec [deg]')
ax.set_title(r'$\sqrt{-\Delta TS}$ map around ' + target_name)
ax.legend(loc='best')
fig.colorbar(contours, ax=ax, label=r'$\sqrt{-\Delta TS}$')
plt.show()

print('Localization result:', loc)
print('Localized position:', getattr(loc, 'dir', None))
print(f'Grid size used: +/-{size_deg:.4f} deg, nside={nside}')
print(f'Delta TS range before clipping: [{delta_ts_raw.min():.2f}, {delta_ts_raw.max():.2f}]')

SyntaxError: invalid syntax (setup_notebook.py, line 11)

ModuleNotFoundError: No module named 'like3'